# Analyse exploratoire — radiations pendant un transit vers Mars

**Projet** : Astro Assistant · NASA Space Apps Challenge 2026
**Auteure** : Ginette · **Relecture** : équipe Astro Assistant

Objectif : caractériser l'environnement radiatif d'un équipage en route vers Mars à partir de **mesures réelles**, pour calibrer le module de dosimétrie (F5) et les seuils d'alerte du moteur.

| Jeu de données | Fichier | Contenu |
|---|---|---|
| NASA PDS — Curiosity RAD (`MSL-M-RAD-3-RDR-V1.0`) | `data/nasa/msl_rad_cruise_dose.csv` | 5 117 mesures du débit de dose à l'intérieur du vaisseau, 06/12/2011 → 13/07/2012 (un jour sur trois) |
| NOAA SWPC — GOES | `data/nasa/goes/integral_protons_*.json` | Flux de protons sur 7 jours (instantané récent) |
| NASA DONKI | `data/nasa/donki/*.json` | Événements à particules solaires (SEP) et éjections de masse coronale (CME) catalogués |

Unités : µGy/h = dose absorbée ; µSv/h = équivalent de dose (dose absorbée × facteur de qualité 3,8 mesuré en croisière, Zeitlin et al., *Science* 2013) ; 1 mSv = 1 000 µSv.

Prérequis : `pip install -e ".[notebooks]"`, puis exécuter depuis le dossier `notebooks/`.

## 1. Chargement des données

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration esthétique des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

DATA_DIR = Path("../data/nasa")

# 1. Chargement de Curiosity RAD (msl_rad_cruise_dose.csv)
df_rad = pd.read_csv(DATA_DIR / "msl_rad_cruise_dose.csv")
print("=== MSL RAD CRUISE DOSE ===")
print(f"Dimensions : {df_rad.shape}")
print("Colonnes :", df_rad.columns.tolist())
display(df_rad.head(3))

# 2. Aperçu de NOAA GOES (integral_protons)
with open(DATA_DIR / "goes/integral_protons_20261114.json", encoding="utf-8") as f:
    goes_raw = json.load(f)
df_goes = pd.DataFrame(goes_raw)
print("\n=== NOAA GOES PROTONS ===")
print(f"Dimensions : {df_goes.shape}")
print("Colonnes :", df_goes.columns.tolist())
display(df_goes.head(3))

## 2. Statistiques descriptives du débit de dose

In [ ]:
# Conversion de la colonne temporelle et tri chronologique
df_rad["utc"] = pd.to_datetime(df_rad["utc"])
df_rad = df_rad.sort_values("utc").reset_index(drop=True)

# Conversion du débit de dose équivalente (µSv/h) en dose journalière (mSv/jour)
# 1 jour = 24h et 1 mSv = 1000 µSv => facteur 24 / 1000 = 0.024
df_rad["dose_msv_day"] = df_rad["dose_equivalent_usv_h"] * 24 / 1000

print(f"Période couverte : du {df_rad['utc'].min()} au {df_rad['utc'].max()}")
print(f"Valeurs manquantes :\n{df_rad.isna().sum()}\n")

# Statistiques descriptives complètes
stats = df_rad[["dose_rate_b_ugy_h", "dose_rate_e_ugy_h", "dose_equivalent_usv_h", "dose_msv_day"]].describe()
display(stats.round(4))

mean_dose_msv_day = df_rad["dose_msv_day"].mean()
median_dose_msv_day = df_rad["dose_msv_day"].median()
max_dose_msv_day = df_rad["dose_msv_day"].max()

print(f"Dose équivalente moyenne : {mean_dose_msv_day:.3f} mSv/jour")
print(f"Dose équivalente médiane (bruit de fond GCR) : {median_dose_msv_day:.3f} mSv/jour")
max_rate_usv_h = df_rad["dose_equivalent_usv_h"].max()
print(f"Débit instantané maximal : {max_rate_usv_h:.0f} µSv/h "
      f"(soit {max_dose_msv_day:.1f} mSv/jour s'il durait 24 h)")

**Lecture.** La distribution est très asymétrique. La **médiane (1,67 mSv/j)** représente le fond cosmique galactique (GCR), présent en permanence ; la **moyenne (2,09 mSv/j)** est tirée vers le haut par quelques tempêtes solaires, pendant lesquelles le débit instantané est multiplié par ~35. Aucune valeur manquante : le CSV a été trié et dédoublonné à l'import.

Le maximum exprimé en mSv/jour est un débit instantané ramené à 24 h : une tempête dure quelques heures, pas une journée entière.

## 3. Évolution dans le temps et pics solaires

In [ ]:
# Seuil empirique pour isoler les tempêtes solaires (SPE) au-dessus du fond galactique (GCR ~ 1.4 - 1.8 mSv/j)
seuil_pic = 2.5
pics_solaires = df_rad[df_rad["dose_msv_day"] > seuil_pic]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_rad["utc"], df_rad["dose_msv_day"], color="#1f77b4", linewidth=0.8, label="Débit de dose (mSv/jour)")
ax.scatter(pics_solaires["utc"], pics_solaires["dose_msv_day"], color="#d62728", s=18, zorder=5, label=f"Pics solaires (> {seuil_pic} mSv/j)")

# Annotation des événements solaires majeurs du transit MSL (Janvier, Mars, Mai 2012)
evenements = [
    ("2012-01-23", "Tempête SPE\nJanv. 2012"),
    ("2012-03-07", "Tempête SPE\nMars 2012 (X5.4)"),
    ("2012-05-17", "Événement GLE\nMai 2012"),
]
for date_str, label in evenements:
    t = pd.to_datetime(date_str, utc=True)
    # Trouver la valeur maximale autour de chaque événement (+/- 4 jours)
    fenetre = df_rad[(df_rad["utc"] >= t - pd.Timedelta(days=2)) & (df_rad["utc"] <= t + pd.Timedelta(days=5))]
    if not fenetre.empty:
        idx_max = fenetre["dose_msv_day"].idxmax()
        row_max = df_rad.loc[idx_max]
        ax.annotate(
            f"{label}\n({row_max['dose_msv_day']:.2f} mSv/j)",
            xy=(row_max["utc"], row_max["dose_msv_day"]),
            xytext=(0, 15),
            textcoords="offset points",
            ha="center",
            fontsize=9,
            fontweight="bold",
            color="#b2182b",
            arrowprops=dict(arrowstyle="->", color="#b2182b", lw=1),
        )

ax.set_title("Évolution du débit de dose équivalente durant le transit Terre-Mars (Curiosity RAD)", fontsize=13, fontweight="bold")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Débit de dose (mSv / jour)")
ax.set_ylim(0, df_rad["dose_msv_day"].max() * 1.22)
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Le fond reste stable autour de 1,6-1,8 mSv/j pendant toute la croisière. Trois épisodes ressortent nettement : **fin janvier 2012**, **début mars 2012** (éruption X5.4) et **mi-mai 2012** (événement GLE détecté jusqu'au sol). Le seuil de 2,5 mSv/j est empirique ; la section 8 vérifie que ces pics correspondent à des tempêtes cataloguées par la NASA.

## 4. Distribution : fond galactique et tempêtes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vue 1 : Échelle complète avec axe Y logarithmique (pour voir la longue traîne des pics solaires)
sns.histplot(df_rad["dose_msv_day"], bins=60, color="#2b5c8f", ax=axes[0])
axes[0].set_yscale("log")
axes[0].axvline(mean_dose_msv_day, color="red", linestyle="--", label=f"Moyenne : {mean_dose_msv_day:.2f} mSv/j")
axes[0].axvline(median_dose_msv_day, color="orange", linestyle=":", label=f"Médiane : {median_dose_msv_day:.2f} mSv/j")
axes[0].set_title("Distribution complète (échelle log sur les effectifs)", fontweight="bold")
axes[0].set_xlabel("Débit de dose (mSv / jour)")
axes[0].set_ylabel("Nombre de mesures (log)")
axes[0].legend()

# Vue 2 : Zoom sur le bruit de fond galactique (GCR < 2.5 mSv/j)
gcr_only = df_rad[df_rad["dose_msv_day"] <= 2.5]["dose_msv_day"]
sns.histplot(gcr_only, bins=40, kde=True, color="#2ca02c", ax=axes[1])
axes[1].axvline(gcr_only.mean(), color="darkgreen", linestyle="--", label=f"Moyenne hors tempête : {gcr_only.mean():.2f} mSv/j")
axes[1].set_title("Zoom sur le bruit de fond cosmique galactique (GCR)", fontweight="bold")
axes[1].set_xlabel("Débit de dose (mSv / jour)")
axes[1].set_ylabel("Nombre de mesures")
axes[1].legend()

plt.tight_layout()
plt.show()

**Lecture.** Hors tempête, la distribution est resserrée et presque symétrique : c'est le régime nominal qu'Astro Assistant doit considérer comme normal. Les tempêtes forment une longue traîne de valeurs rares mais très élevées, d'où l'échelle logarithmique.

## 5. Dose cumulée et projection sur la mission

In [ ]:
DUREE_TRANSIT_JOURS = 210
LIMITE_NASA_MSV = 600.0

# 1. Extrapolation basée sur le débit moyen global (incluant les tempêtes solaires)
dose_210j_moyenne = mean_dose_msv_day * DUREE_TRANSIT_JOURS

# 2. Intégration trapézoïdale exacte sur la durée réelle du jeu de données, puis mise à l'échelle à 210j
duree_reelle_heures = (df_rad["utc"].max() - df_rad["utc"].min()).total_seconds() / 3600
# Conversion en heures depuis le début pour intégration numérique
heures_ecoulees = (df_rad["utc"] - df_rad["utc"].min()).dt.total_seconds() / 3600
dose_totale_observee_msv = np.trapezoid(df_rad["dose_equivalent_usv_h"], heures_ecoulees) / 1000
dose_210j_integree = dose_totale_observee_msv * (DUREE_TRANSIT_JOURS * 24 / duree_reelle_heures)

pct_limite_moy = (dose_210j_moyenne / LIMITE_NASA_MSV) * 100
pct_limite_int = (dose_210j_integree / LIMITE_NASA_MSV) * 100

print(f"Durée réelle observée : {duree_reelle_heures / 24:.1f} jours (dose intégrée = {dose_totale_observee_msv:.2f} mSv)")
print(f"Dose cumulée extrapolée à 210 jours (via moyenne)     : {dose_210j_moyenne:.2f} mSv ({pct_limite_moy:.1f}% de la limite NASA)")
print(f"Dose cumulée extrapolée à 210 jours (via intégration) : {dose_210j_integree:.2f} mSv ({pct_limite_int:.1f}% de la limite NASA)")
print(f"Aller-retour Mars (420 jours, sans surface)           : {dose_210j_integree * 2:.2f} mSv ({pct_limite_int * 2:.1f}% de la limite NASA)")

# Graphique comparatif
fig, ax = plt.subplots(figsize=(9, 4.5))
categories = [
    f"Transit observé\n({duree_reelle_heures/24:.0f} jours)",
    "Aller Terre-Mars\n(extrapolé 210 jours)",
    "Aller-Retour\n(extrapolé 420 jours)",
    "Limite Carrière\nNASA (600 mSv)"
]
valeurs = [dose_totale_observee_msv, dose_210j_integree, dose_210j_integree * 2, LIMITE_NASA_MSV]
couleurs = ["#4c72b0", "#dd8452", "#c44e52", "#55a868"]

bars = ax.bar(categories, valeurs, color=couleurs, width=0.55)
ax.axhline(LIMITE_NASA_MSV, color="#2e7d32", linestyle="--", linewidth=1.5, label="Limite NASA : 600 mSv")

for bar, val in zip(bars, valeurs):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 12, f"{val:.1f} mSv\n({val/LIMITE_NASA_MSV*100:.0f}%)",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold")

ax.set_ylim(0, LIMITE_NASA_MSV * 1.2)
ax.set_ylabel("Dose cumulée (mSv)")
ax.set_title("Projection de la dose de radiation sur 210 jours vs Limite de carrière NASA", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Sur les 220 jours observés, l'équipage aurait reçu environ **420 mSv**. Rapporté à un aller de **210 jours**, on obtient **≈ 400 à 440 mSv**, soit **67 à 73 % de la limite de carrière NASA (600 mSv)** pour un seul trajet. Un aller-retour sans séjour en surface **dépasse la limite (≈ 134 %)**, sans même compter la dose reçue lors de missions précédentes.

Limites : un jour sur trois seulement est mesuré (l'intégration interpole entre les jours) ; une seule phase du cycle solaire est couverte ; le blindage de Curiosity n'est pas celui d'un habitat habité.

## 6. Cohérence des deux détecteurs (B silicium, E plastique)

In [ ]:
corr_pearson = df_rad["dose_rate_b_ugy_h"].corr(df_rad["dose_rate_e_ugy_h"], method="pearson")
corr_spearman = df_rad["dose_rate_b_ugy_h"].corr(df_rad["dose_rate_e_ugy_h"], method="spearman")

print(f"Corrélation de Pearson  (B vs E) : r = {corr_pearson:.4f}")
print(f"Corrélation de Spearman (B vs E) : rho = {corr_spearman:.4f}")
calme_b = df_rad[df_rad["dose_rate_b_ugy_h"] < 30]
r_hors_tempete = calme_b["dose_rate_b_ugy_h"].corr(calme_b["dose_rate_e_ugy_h"])
print(f"Pearson hors tempête (B < 30 µGy/h) : r = {r_hors_tempete:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Nuage de points global
sns.regplot(
    data=df_rad,
    x="dose_rate_b_ugy_h",
    y="dose_rate_e_ugy_h",
    scatter_kws={"alpha": 0.35, "s": 15, "color": "#1f77b4"},
    line_kws={"color": "red", "lw": 1.5},
    ax=axes[0],
)
axes[0].set_title(f"Corrélation Détecteur B vs Détecteur E (r = {corr_pearson:.4f})", fontweight="bold")
axes[0].set_xlabel("Détecteur B — Silicium (µGy/h)")
axes[0].set_ylabel("Détecteur E — Plastique (µGy/h)")

# Zoom sur le régime calme (GCR < 30 µGy/h)
calme = df_rad[df_rad["dose_rate_b_ugy_h"] < 30]
r_calme = calme["dose_rate_b_ugy_h"].corr(calme["dose_rate_e_ugy_h"])
sns.regplot(
    data=calme,
    x="dose_rate_b_ugy_h",
    y="dose_rate_e_ugy_h",
    scatter_kws={"alpha": 0.35, "s": 15, "color": "#2ca02c"},
    line_kws={"color": "darkgreen", "lw": 1.5},
    ax=axes[1],
)
axes[1].set_title(f"Zoom hors tempête solaire (r = {r_calme:.4f})", fontweight="bold")
axes[1].set_xlabel("Détecteur B — Silicium (µGy/h)")
axes[1].set_ylabel("Détecteur E — Plastique (µGy/h)")

plt.tight_layout()
plt.show()

**Lecture.** La corrélation de Pearson très élevée (**r ≈ 0,99**) est surtout portée par les tempêtes, pendant lesquelles les deux détecteurs montent ensemble. La corrélation de Spearman, bien plus faible (**ρ ≈ 0,31**), montre qu'en régime calme leurs petites variations sont peu liées : elles relèvent surtout du bruit statistique de comptage. Le moteur s'appuie sur le détecteur **E** (plastique, proche du tissu humain), qui sert à calculer l'équivalent de dose.

## 7. Alerte précoce : flux de protons NOAA GOES

In [ ]:
df_goes["time_tag"] = pd.to_datetime(df_goes["time_tag"])

# Filtrer le canal >= 10 MeV (référence NOAA pour les tempêtes de radiations solaires S1)
goes_10mev = df_goes[df_goes["energy"] == ">=10 MeV"].sort_values("time_tag").copy()
SEUIL_PFU = 10.0
depassements = goes_10mev[goes_10mev["flux"] >= SEUIL_PFU]

print("=== ANALYSE NOAA GOES (>=10 MeV) ===")
print(f"Période GOES : du {goes_10mev['time_tag'].min()} au {goes_10mev['time_tag'].max()}")
print(f"Nombre total de mesures (>=10 MeV) : {len(goes_10mev)}")
print(f"Nombre de dépassements du seuil de {SEUIL_PFU} pfu : {len(depassements)}")
print(f"Flux maximal observé : {goes_10mev['flux'].max():.4f} pfu")

if not depassements.empty:
    display(depassements[["time_tag", "satellite", "flux", "energy"]].head(10))
else:
    print("Aucun dépassement >= 10 pfu sur cette fenêtre temporelle (conditions solaires calmes).")

# Graphique du flux de protons GOES (>=10 MeV) vs seuil d'alerte 10 pfu
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(goes_10mev["time_tag"], goes_10mev["flux"], color="#1f77b4", lw=1, label="Flux protons >= 10 MeV (pfu)")
ax.axhline(SEUIL_PFU, color="#d62728", linestyle="--", lw=1.5, label="Seuil tempête solaire NOAA S1 (10 pfu)")
if not depassements.empty:
    ax.scatter(depassements["time_tag"], depassements["flux"], color="red", s=20, zorder=5, label="Dépassements >= 10 pfu")

ax.set_yscale("log")
ax.set_title("Flux intégral de protons solaires NOAA GOES (>= 10 MeV) vs Seuil 10 pfu", fontweight="bold")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Flux de protons (pfu, échelle log)")
ax.legend()
plt.tight_layout()
plt.show()

# Synthèse des événements historiques DONKI (2017 et 2024)
print("\n=== FICHIERS NASA DONKI DISPONIBLES ===")
for json_file in sorted((DATA_DIR / "donki").glob("*.json")):
    with open(json_file, encoding="utf-8") as f:
        events = json.load(f)
    print(f"- {json_file.name} : {len(events)} événements enregistrés")

**Lecture.** Sur la semaine récupérée, le flux ≥ 10 MeV reste très en dessous du seuil de tempête S1 (10 pfu) : conditions solaires calmes. Ce flux sert d'**alerte précoce** : il augmente avant la dose mesurée à bord, ce qui laisse le temps de rejoindre l'abri (F5.4).

## 8. Validation croisée : pics RAD et tempêtes solaires NASA DONKI

In [ ]:
# Chargement des événements à particules solaires (SEP) DONKI pendant la croisière
sep_rows = []
for json_file in sorted((DATA_DIR / "donki").glob("sep_*.json")):
    for event in json.loads(json_file.read_text(encoding="utf-8")):
        sep_rows.append({"sep_id": event["sepID"], "debut": event["eventTime"]})
df_sep = pd.DataFrame(sep_rows).drop_duplicates("sep_id")
df_sep["debut"] = pd.to_datetime(df_sep["debut"], utc=True)
debut_croisiere, fin_croisiere = df_rad["utc"].min(), df_rad["utc"].max()
sep_croisiere = df_sep[(df_sep["debut"] >= debut_croisiere) & (df_sep["debut"] <= fin_croisiere)]
sep_croisiere = sep_croisiere.sort_values("debut")
print(f"SEP catalogués par la NASA pendant la croisière : {len(sep_croisiere)}")

# Épisodes de pics RAD : jours au-dessus du seuil, regroupés s'ils sont espacés de moins de 4 jours
jours_pics = sorted(pics_solaires["utc"].dt.normalize().unique())
episodes = []
for jour in jours_pics:
    if episodes and (jour - episodes[-1][-1]) <= pd.Timedelta(days=4):
        episodes[-1].append(jour)
    else:
        episodes.append([jour])

lignes = []
for jours in episodes:
    fenetre = df_rad[(df_rad["utc"] >= jours[0]) & (df_rad["utc"] < jours[-1] + pd.Timedelta(days=1))]
    pic = fenetre.loc[fenetre["dose_msv_day"].idxmax()]
    # Un SEP explique le pic s'il a commencé au plus 3 jours avant (propagation et durée de l'événement)
    candidats = sep_croisiere[
        (sep_croisiere["debut"] <= pic["utc"]) & (sep_croisiere["debut"] >= pic["utc"] - pd.Timedelta(days=3))
    ]
    lignes.append({
        "debut_episode": jours[0].date(),
        "pic_utc": pic["utc"].strftime("%Y-%m-%d %H:%M"),
        "pic_msv_jour": round(pic["dose_msv_day"], 2),
        "sep_nasa": ", ".join(candidats["sep_id"].str[:16]) or "aucun",
    })
df_episodes = pd.DataFrame(lignes)
expliques = int((df_episodes["sep_nasa"] != "aucun").sum())
print(f"Épisodes de pics RAD : {len(df_episodes)}, dont {expliques} précédés d'un SEP NASA "
      f"({expliques / len(df_episodes):.0%})")
display(df_episodes)

# Courbe RAD avec les débuts des SEP en surimpression
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_rad["utc"], df_rad["dose_msv_day"], color="#1f77b4", lw=0.8, label="Débit de dose RAD (mSv/jour)")
for i, debut in enumerate(sep_croisiere["debut"]):
    ax.axvline(debut, color="#d62728", alpha=0.35, lw=1, label="Début d'un SEP (NASA DONKI)" if i == 0 else None)
ax.set_yscale("log")
ax.set_title("Curiosity RAD vs événements à particules solaires catalogués par la NASA", fontweight="bold")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Débit de dose (mSv / jour, échelle log)")
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** __DONKI_RESULT__

## 9. Conclusions pour Astro Assistant

1. **Fond de référence** : ≈ 1,67 mSv/jour (médiane). C'est le régime normal, affiché en vert sur l'écran de dosimétrie.
2. **Seuil de tempête** : au-delà de ≈ 2,5 mSv/jour (~1,5 fois le fond), un événement solaire est en cours. Il doit déclencher une alerte et le protocole d'abri **PRT-RAD-02**.
3. **Alerte précoce** : un flux GOES ≥ 10 MeV au-dessus de 10 pfu (seuil NOAA S1) précède la hausse de dose ; il doit être surveillé en continu.
4. **Enjeu de mission** : un aller vers Mars consomme à lui seul 67 à 73 % de la limite de carrière ; le suivi de la **dose cumulée de carrière** compte autant que l'alerte en temps réel.
5. **Validation** : __CONCLUSION_DONKI__